# Wagon Eye — Phase 1 standalone wagon counter

> **Optional local-debugging aid — not the production path.** This notebook is
> a convenience for interactively exploring the standalone Phase-1 package on a
> dev machine or in any Jupyter install. Production EC2 deployment runs the
> pipeline headless via `python -m orchestrator.master_runner` (see the
> top-level `DEPLOYMENT.md`) or the `run_global_count.py` CLI — **no notebook or
> Jupyter runtime is required.**

End-to-end notebook for the **standalone** Phase-1 wagon counting package (the contents of this `wagon_count/` folder).

This notebook will:

1. Verify the runtime environment (Python, GPU, ultralytics).
2. Install dependencies from `requirements.txt`.
3. Confirm the 4 input videos and 4 `.pt` models are in place.
4. Run the full Phase-1 pipeline (per-camera tracking → master classification → cross-camera fusion → overlay videos → per-wagon frame extraction).
5. Inspect the final `GlobalTrainState`.
6. Preview a few extracted frames.

> **Phase 1 only.** No doors, no damage, no OCR, no PDF, no email, no S3.
>
> Just global wagon segmentation + ENGINE/WAGON/BRAKE\_VAN classification + processed overlay videos + per-wagon frame folders.

Run cells **top-to-bottom**. Each cell either prints a confirmation or raises early if something is missing.

## 1. Environment check

Python version, working directory, GPU availability. Phase 1 will run on CPU but YOLO inference is ~10× faster on GPU.

In [ ]:
import sys, os, platform, shutil
print(f"Python   : {sys.version.split()[0]}")
print(f"Platform : {platform.system()} {platform.release()}")
print(f"CWD      : {os.getcwd()}")

try:
    import torch
    print(f"torch    : {torch.__version__}  cuda={torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"         : {torch.cuda.get_device_name(0)}")
except ImportError:
    print("torch    : (not installed yet — the next cell will install it)")

## 2. Install Python dependencies

Installs from the package's own `requirements.txt`. Already-installed packages are skipped silently. Re-run after kernel restarts if needed.

In [ ]:
%pip install -q -r requirements.txt

## 3. Locate the package root and confirm layout

`PKG_ROOT` is the folder that contains `run_global_count.py`. If you launched the notebook from inside `wagon_count/`, the auto-detect below is correct.

In [ ]:
PKG_ROOT = os.path.abspath(os.getcwd())
if not os.path.isfile(os.path.join(PKG_ROOT, "run_global_count.py")):
    candidate = os.path.join(PKG_ROOT, "wagon_count")
    if os.path.isfile(os.path.join(candidate, "run_global_count.py")):
        PKG_ROOT = candidate

print(f"PKG_ROOT = {PKG_ROOT}\n")

expected_files = [
    "run_global_count.py",
    "global_train_state.py",
    "tracker_engine.py",
    "global_alignment.py",
    "video_segmenter.py",
    "requirements.txt",
]
expected_dirs = ["inputs", "models"]

missing = []
for f in expected_files:
    p = os.path.join(PKG_ROOT, f)
    ok = os.path.isfile(p)
    print(f"  {'✓' if ok else '✗'} {f}")
    if not ok:
        missing.append(f)
for d in expected_dirs:
    p = os.path.join(PKG_ROOT, d)
    ok = os.path.isdir(p)
    print(f"  {'✓' if ok else '✗'} {d}/")
    if not ok:
        missing.append(d)

assert not missing, f"Required paths missing: {missing}. Run this notebook from inside the wagon_count/ folder."

# Make package modules importable for later cells
if PKG_ROOT not in sys.path:
    sys.path.insert(0, PKG_ROOT)
os.chdir(PKG_ROOT)
print("\n✓ Package layout OK")

## 4. Confirm models and videos are in place

Hard-error NOW if any required file is missing. Phase-1 requires all 4 models and all 4 videos.

In [ ]:
REQUIRED_MODELS = [
    "right_up_wagon_gap.pt",   # RIGHT_UP  (master)
    "left_up_wagon_gap.pt",    # LEFT_UP
    "top_gap.pt",              # RIGHT_UP_TOP + LEFT_UP_TOP
    "side_classification.pt",  # RIGHT_UP only -- ENGINE / WAGON / BRAKE_VAN
]
REQUIRED_VIDEOS = ["right_up.mp4", "left_up.mp4", "right_up_top.mp4", "left_up_top.mp4"]

MODELS_DIR = os.path.join(PKG_ROOT, "models")
INPUTS_DIR = os.path.join(PKG_ROOT, "inputs")

all_ok = True
print(f"models/  ({MODELS_DIR})")
for m in REQUIRED_MODELS:
    p = os.path.join(MODELS_DIR, m)
    ok = os.path.isfile(p)
    mb = os.path.getsize(p) / (1024 * 1024) if ok else 0
    print(f"  {'✓' if ok else '✗'} {m:<26} ({mb:5.1f} MB)")
    all_ok = all_ok and ok

print(f"\ninputs/  ({INPUTS_DIR})")
for v in REQUIRED_VIDEOS:
    p = os.path.join(INPUTS_DIR, v)
    ok = os.path.isfile(p)
    mb = os.path.getsize(p) / (1024 * 1024) if ok else 0
    print(f"  {'✓' if ok else '✗'} {v:<24} ({mb:6.1f} MB)")
    all_ok = all_ok and ok

print("\n" + ("✅ All inputs and models in place" if all_ok else "❌ Missing files — fix before continuing"))
assert all_ok, "Drop the missing files into models/ and inputs/ and re-run this cell."

## 5. Configure the run

Defaults match the CLI defaults of `run_global_count.py`. Edit only if you need to override a threshold.

In [ ]:
OUTPUT_DIR = os.path.join(PKG_ROOT, "results")

# Detection thresholds
SIDE_CONFIDENCE = 0.4         # YOLO conf for right_up_wagon_gap.pt + left_up_wagon_gap.pt
TOP_CONFIDENCE  = 0.4         # YOLO conf for top_gap.pt
CLASSIFICATION_SAMPLES = 5    # frames per segment voted in side_classification.pt

# Min-bbox-height / frame-height filters (camera geometry differs):
#   SIDE cameras see the gap as a TALL vertical strip -> filter at 0.35
#   TOP cameras see the gap as a THIN horizontal strip -> filter at 0.05
# If your top videos still don't show gap bboxes, lower TOP_MIN_HEIGHT_RATIO
# further (e.g. 0.02) -- the tracker's hit/miss persistence rule will still
# filter noise at the track level.
SIDE_MIN_HEIGHT_RATIO = 0.35
TOP_MIN_HEIGHT_RATIO  = 0.05

# Fusion thresholds (how strict we are about inserting a gap RIGHT_UP missed)
FUSE_MIN_SUPPORT = 2          # need >= 2 support cameras to vote in a missed gap
FUSE_MAX_SPREAD  = 1.5        # seconds: max time spread within a fusion cluster
FUSE_MIN_CONF    = 0.4        # min mean confidence to insert a fused gap

# Output toggles
RENDER_VIDEOS  = True         # set False to skip overlay rendering (faster runs)
EXTRACT_FRAMES = True         # set False to skip per-wagon JPEG extraction
EVERY_NTH_FRAME = 1           # keep 1 of every N frames (use 5 for sparser sampling)

VERBOSE = True

print(f"OUTPUT_DIR             : {OUTPUT_DIR}")
print(f"side_confidence        : {SIDE_CONFIDENCE}")
print(f"top_confidence         : {TOP_CONFIDENCE}")
print(f"side_min_height_ratio  : {SIDE_MIN_HEIGHT_RATIO}")
print(f"top_min_height_ratio   : {TOP_MIN_HEIGHT_RATIO}")
print(f"classification_samples : {CLASSIFICATION_SAMPLES}")
print(f"fuse_min_support       : {FUSE_MIN_SUPPORT}")
print(f"fuse_max_spread        : {FUSE_MAX_SPREAD}s")
print(f"fuse_min_confidence    : {FUSE_MIN_CONF}")
print(f"render_videos          : {RENDER_VIDEOS}")
print(f"extract_frames         : {EXTRACT_FRAMES}  (every {EVERY_NTH_FRAME} frame)")

## 6. Run the full Phase-1 pipeline

Two ways to run — pick **one**:

* **6a (recommended for notebooks)** — call the pipeline functions directly. Keeps the intermediate `tracks` / `state` objects in memory so you can inspect them in later cells.
* **6b** — invoke `run_global_count.py` as a subprocess. Same outputs, but you lose the in-memory objects.

### 6a. In-process run (recommended)

In [ ]:
import time
from global_train_state import (
    GlobalTrainState, LocalCameraTracks, SegmentClass,
    MASTER_CAMERA, ALL_CAMERAS,
    CAMERA_RIGHT_UP, CAMERA_LEFT_UP, CAMERA_RIGHT_UP_TOP, CAMERA_LEFT_UP_TOP,
    summarize_state,
)
from tracker_engine import GapTracker, MasterClassifier, segments_from_gaps
import global_alignment as ga
import video_segmenter as vs
import json

os.makedirs(OUTPUT_DIR, exist_ok=True)
PROCESSED_VIDEOS_DIR = os.path.join(OUTPUT_DIR, "processed_videos")
FRAMES_ROOT          = os.path.join(OUTPUT_DIR, "frames")
os.makedirs(PROCESSED_VIDEOS_DIR, exist_ok=True)
os.makedirs(FRAMES_ROOT, exist_ok=True)

# Two separate side gap models -- one per side camera.
model_paths = {
    "right_up_gap":        os.path.join(MODELS_DIR, "right_up_wagon_gap.pt"),
    "left_up_gap":         os.path.join(MODELS_DIR, "left_up_wagon_gap.pt"),
    "top_gap":             os.path.join(MODELS_DIR, "top_gap.pt"),
    "side_classification": os.path.join(MODELS_DIR, "side_classification.pt"),
}
video_paths = {
    CAMERA_RIGHT_UP:     os.path.join(INPUTS_DIR, "right_up.mp4"),
    CAMERA_LEFT_UP:      os.path.join(INPUTS_DIR, "left_up.mp4"),
    CAMERA_RIGHT_UP_TOP: os.path.join(INPUTS_DIR, "right_up_top.mp4"),
    CAMERA_LEFT_UP_TOP:  os.path.join(INPUTS_DIR, "left_up_top.mp4"),
}

# Pick the right gap model + filter settings for each camera.
def _gap_model_for(cam):
    if cam == CAMERA_RIGHT_UP:
        return model_paths["right_up_gap"]
    if cam == CAMERA_LEFT_UP:
        return model_paths["left_up_gap"]
    return model_paths["top_gap"]    # both top cameras share top_gap.pt

def _filters_for(cam):
    is_top = cam in (CAMERA_RIGHT_UP_TOP, CAMERA_LEFT_UP_TOP)
    return (
        TOP_CONFIDENCE if is_top else SIDE_CONFIDENCE,
        TOP_MIN_HEIGHT_RATIO if is_top else SIDE_MIN_HEIGHT_RATIO,
    )

t_start = time.time()

# ---- Step 1: per-camera tracking ----
print("=" * 70)
print("  STEP 1  Per-camera gap tracking")
print("=" * 70)
tracks = {}
for cam in ALL_CAMERAS:
    model = _gap_model_for(cam)
    conf, mhr = _filters_for(cam)
    tracker = GapTracker(
        camera_id=cam, model_path=model,
        confidence=conf, min_height_ratio=mhr,
        verbose=VERBOSE,
    )
    tracks[cam] = tracker.process_video(video_paths[cam], keep_raw_detections=True)

print("\nLocal counts after Step 1:")
for cam in ALL_CAMERAS:
    t = tracks[cam]
    print(f"  {cam:<14}  wagons={t.local_wagon_count:>3}  gaps={len(t.gaps):>3}  fps={t.fps:.2f}  frames={t.total_frames}")

# ---- Step 2: master classification ----
print("\n" + "=" * 70)
print("  STEP 2  RIGHT_UP master classification (ENGINE / WAGON / BRAKE_VAN)")
print("=" * 70)
master = tracks[CAMERA_RIGHT_UP]
pre_segments = segments_from_gaps(master.gaps, master.total_frames)
classifier = MasterClassifier(model_paths["side_classification"], num_samples=CLASSIFICATION_SAMPLES, verbose=VERBOSE)
initial_classifications = classifier.classify_segments(master.video_path, pre_segments)

# ---- Step 3: cross-camera fusion ----
print("\n" + "=" * 70)
print("  STEP 3  Cross-camera gap fusion")
print("=" * 70)
support = [tracks[c] for c in ALL_CAMERAS if c != CAMERA_RIGHT_UP]
fuse_cfg = dict(ga.PHASE1_DEFAULTS)
fuse_cfg.update({
    "insert_min_support": FUSE_MIN_SUPPORT,
    "insert_max_spread_sec": FUSE_MAX_SPREAD,
    "insert_min_confidence": FUSE_MIN_CONF,
})
state = ga.assemble_global_train_state(
    master_tracks=master,
    support_tracks=support,
    initial_classifications=initial_classifications,
    config=fuse_cfg,
    verbose=VERBOSE,
)

# ---- Step 4: write JSON ----
state_json_path = os.path.join(OUTPUT_DIR, "global_train_state.json")
with open(state_json_path, "w", encoding="utf-8") as f:
    f.write(state.to_json())

tracking_dump = {
    cam: tracks[cam].to_dict(include_classifications=(cam == CAMERA_RIGHT_UP))
    for cam in ALL_CAMERAS
}
if initial_classifications:
    tracking_dump[CAMERA_RIGHT_UP]["pre_fusion_classifications"] = [c.to_dict() for c in initial_classifications]
with open(os.path.join(OUTPUT_DIR, "per_camera_tracking.json"), "w", encoding="utf-8") as f:
    json.dump(tracking_dump, f, indent=2)

# ---- Step 5: overlay videos ----
if RENDER_VIDEOS:
    print("\n" + "=" * 70)
    print("  STEP 5  Overlay videos")
    print("=" * 70)
    for cam in ALL_CAMERAS:
        out_mp4 = os.path.join(PROCESSED_VIDEOS_DIR, f"{cam}_processed.mp4")
        try:
            vs.render_processed_video(
                local_tracks=tracks[cam],
                state=state,
                output_path=out_mp4,
                draw_raw_detections=True,
                verbose=VERBOSE,
            )
        except Exception as e:
            print(f"  ⚠ render failed for {cam}: {e}")
            state.add_note(f"render_failed:{cam}:{e}")

# ---- Step 6: per-wagon frame extraction ----
if EXTRACT_FRAMES:
    print("\n" + "=" * 70)
    print("  STEP 6  Per-wagon frame extraction")
    print("=" * 70)
    for cam in ALL_CAMERAS:
        try:
            vs.extract_wagon_frames(
                local_tracks=tracks[cam],
                state=state,
                output_root=FRAMES_ROOT,
                every_nth_frame=EVERY_NTH_FRAME,
                verbose=VERBOSE,
            )
        except Exception as e:
            print(f"  ⚠ extraction failed for {cam}: {e}")
            state.add_note(f"frame_extraction_failed:{cam}:{e}")

# Re-dump JSON with any notes from failed render/extraction steps
with open(state_json_path, "w", encoding="utf-8") as f:
    f.write(state.to_json())

elapsed = time.time() - t_start
print("\n" + summarize_state(state))
print(f"\n  total elapsed : {elapsed:.1f}s")
print(f"  output root   : {OUTPUT_DIR}")

### 6b. Subprocess run (alternative)

Same result as 6a, but invokes `run_global_count.py` as a subprocess. Use this if you'd rather mirror the AWS CLI invocation exactly. **Skip this cell if you already ran 6a.**

In [ ]:
import subprocess

cmd = [
    sys.executable, os.path.join(PKG_ROOT, "run_global_count.py"),
    "--output", OUTPUT_DIR,
    "--side-confidence", str(SIDE_CONFIDENCE),
    "--top-confidence",  str(TOP_CONFIDENCE),
    "--classification-samples", str(CLASSIFICATION_SAMPLES),
    "--fuse-min-support", str(FUSE_MIN_SUPPORT),
    "--fuse-max-spread",  str(FUSE_MAX_SPREAD),
    "--fuse-min-conf",    str(FUSE_MIN_CONF),
    "--every-nth-frame",  str(EVERY_NTH_FRAME),
]
if not RENDER_VIDEOS:  cmd.append("--no-videos")
if not EXTRACT_FRAMES: cmd.append("--no-frames")

print("Running:", " ".join(cmd), "\n")
proc = subprocess.run(cmd, cwd=PKG_ROOT, capture_output=True, text=True, timeout=7200)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print("\n--- STDERR (last 2000) ---\n" + proc.stderr[-2000:])
print(f"\nexit code: {proc.returncode}")

## 7. Inspect the GlobalTrainState

(Only runs after cell 6a, which keeps `state` in memory.)

In [ ]:
print(f"Master camera         : {state.master_camera}")
print(f"Master fps            : {state.master_fps:.2f}")
print(f"Master total frames   : {state.master_total_frames}")
print(f"Total wagons          : {state.total_wagons}")
print(f"  engines             : {state.engine_count}")
print(f"  regular wagons      : {state.regular_wagon_count}")
print(f"  brake vans          : {state.brake_van_count}")
print(f"Corrections applied   : {len(state.corrections_applied)}")
print(f"Fallback used         : {state.fallback_used}")
print()
print("-" * 78)
print(f"  {'GID':<8} {'CLASS':<10} {'start_s':>8} {'end_s':>8} {'frames':>14}  split_from")
print("-" * 78)
for w in state.wagons:
    print(f"  {w.global_id:<8} {w.classification:<10} "
          f"{w.start_time:8.2f} {w.end_time:8.2f} "
          f"{w.start_frame_master:>6}-{w.end_frame_master:<6}  "
          f"{w.split_from_global_id or '—'}")
print("-" * 78)
if state.corrections_applied:
    print("\nGaps inserted by support cameras:")
    for c in state.corrections_applied:
        print(f"  + t={c.inserted_at_master_time:.2f}s  f={c.inserted_at_master_frame}  "
              f"supports={'/'.join(c.supporting_cameras)}  conf={c.mean_confidence:.2f}  "
              f"spread={c.time_spread_sec:.2f}s")

### 7a. Show the raw JSON (first 4 wagons)

In [ ]:
import json
doc = state.to_dict()
preview = {k: v for k, v in doc.items() if k != "wagons"}
preview["wagons_first_4"] = doc["wagons"][:4]
print(json.dumps(preview, indent=2))

## 8. Preview extracted frames

Show one thumbnail from each camera's view of `GW_1` (the engine, usually) and a mid-train wagon. Useful sanity-check that the same `GW_n` id across cameras really does correspond to the same physical wagon.

In [ ]:
import glob
from IPython.display import Image, display, HTML, Markdown

if not state.wagons:
    display(Markdown("_No wagons in state — skip._"))
else:
    # First wagon and a middle wagon
    mid_idx = max(0, len(state.wagons) // 2)
    sample_gw_ids = sorted({state.wagons[0].global_id, state.wagons[mid_idx].global_id})

    for gw_id in sample_gw_ids:
        display(Markdown(f"### {gw_id} — one preview frame per camera"))
        html_rows = '<table style="width:100%"><tr>'
        for cam in ALL_CAMERAS:
            cam_dir = os.path.join(FRAMES_ROOT, cam, gw_id)
            jpgs = sorted(glob.glob(os.path.join(cam_dir, "*.jpg")))
            if not jpgs:
                html_rows += f'<td style="text-align:center"><b>{cam}</b><br><i>(no frames)</i></td>'
                continue
            mid_jpg = jpgs[len(jpgs) // 2]
            rel = os.path.relpath(mid_jpg, PKG_ROOT)
            html_rows += (
                f'<td style="text-align:center">'
                f'<b>{cam}</b><br>'
                f'<img src="{rel}" style="max-width:240px;max-height:160px"><br>'
                f'<small>{os.path.basename(mid_jpg)}</small>'
                f'</td>'
            )
        html_rows += "</tr></table>"
        display(HTML(html_rows))

## 9. Preview an overlay video clip

Play the first overlay video inline (works in any Jupyter install). Skip if you don't need a preview.

In [ ]:
from IPython.display import Video, Markdown, display

video_to_show = os.path.join(PROCESSED_VIDEOS_DIR, "RIGHT_UP_processed.mp4")
if os.path.exists(video_to_show):
    display(Markdown(f"**Preview:** `{os.path.relpath(video_to_show, PKG_ROOT)}`"))
    display(Video(video_to_show, embed=True, width=720))
else:
    display(Markdown("_Run cell 6a (with RENDER_VIDEOS=True) first to generate overlay videos._"))

## 10. Cleanup & tips

* `results/` — all Phase-1 outputs. Safe to delete between runs.
* `results/frames/` — large; consider increasing `EVERY_NTH_FRAME` or zipping for transport.
* Phase-2 (door state / damage / OCR / loaded-empty / report) will consume `results/frames/<CAMERA>/<GW_n>/` directly. Same `GW_n` across cameras means each downstream stage gets a synchronized starting point.

### Running from a shell instead of a notebook

```bash
cd wagon_count
python run_global_count.py
```

### Re-zip the package for AWS

From the directory **above** `wagon_count/`:

```powershell
# Windows PowerShell
Compress-Archive -Path wagon_count -DestinationPath wagon_count.zip -Force
```

```bash
# Linux/macOS
zip -r wagon_count.zip wagon_count -x 'wagon_count/results/*' \
                                   -x 'wagon_count/inputs/*' \
                                   -x 'wagon_count/models/*'
```